# Reproduce Numbers-page figures from the Divine Discourses data pack

Evidence label for everything below: **Verified (reproduction)**. Each check recomputes a figure
the site publishes, using only the downloadable pack (no site code, no corpus access).

Pack: `divinediscourses-data-v<version>.tar.gz`, from <https://divinediscourses.org/export>. Run from a clone of the repository, the notebook uses the current release named in `data/version.json`; elsewhere set `VERSION` by hand. CI runs it on every pull request.
The underlying morphology is the Leeds Quranic Arabic Corpus v0.4 (GPL); see `NOTICE-LEEDS.txt` in the pack.
A check that fails means the pack and the page disagree, which is a bug to report, not a result.

In [ ]:
import io, tarfile, urllib.request, hashlib
import pandas as pd

import json, os, pathlib
# Run from anywhere inside a clone: find the repository root (the folder
# holding data/version.json) and use its current release. Outside a clone,
# REPO is None and the pinned VERSION below is fetched from the site.
REPO = next((d for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (d / "data/version.json").exists()), None)
VERSION = json.load(open(REPO / "data/version.json"))["version"] if REPO else "1.7.0"
URL = f"https://divinediscourses.org/data/exports/divinediscourses-data-v{VERSION}.tar.gz"
LOCAL = REPO / f"data/exports/divinediscourses-data-v{VERSION}.tar.gz" if REPO else None

blob = LOCAL.read_bytes() if LOCAL and LOCAL.exists() else urllib.request.urlopen(URL).read()
print("pack bytes:", len(blob), "sha256:", hashlib.sha256(blob).hexdigest()[:16], "...")

tar = tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz")
read = lambda name: pd.read_csv(tar.extractfile(name))
roots = read("root-frequencies.csv")
surahs = read("surah-stats.csv")
by_surah = read("root-surah-counts.csv")

## Check 1: rooted word-units (Numbers page, "rooted tokens")
Every rooted token belongs to exactly one root, so the per-root counts must sum to the number of rooted tokens.

In [ ]:
PUBLISHED_ROOTED_TOKENS = 49967      # data/numbers.json totals.rootedTokens
PUBLISHED_ROOTS = 1642               # distinct roots

assert roots["totalCount"].sum() == PUBLISHED_ROOTED_TOKENS, roots["totalCount"].sum()
assert len(roots) == PUBLISHED_ROOTS, len(roots)
print("OK: sum of per-root counts =", roots["totalCount"].sum(), "over", len(roots), "roots")

## Check 2: total word-units, two independent tables
Surah token counts and per-root counts come from different export tables; the rooted share must be consistent.

In [ ]:
PUBLISHED_TOKENS = 77429
assert surahs["tokenCount"].sum() == PUBLISHED_TOKENS, surahs["tokenCount"].sum()
assert surahs["verseCount"].sum() == 6236
rooted_share = PUBLISHED_ROOTED_TOKENS / PUBLISHED_TOKENS
print(f"OK: {PUBLISHED_TOKENS:,} word-units in 114 surahs; {rooted_share:.2%} carry a root")

## Check 3: the five most frequent roots (Numbers page, "top roots")

In [ ]:
PUBLISHED_TOP5 = [("Alh", 2851), ("qwl", 1722), ("kwn", 1390), ("rbb", 980), ("Amn", 879)]
top5 = list(roots.sort_values(["totalCount", "root"], ascending=[False, True]).head(5)[["root", "totalCount"]].itertuples(index=False, name=None))
assert top5 == PUBLISHED_TOP5, top5
print("OK:", top5)

## Check 4: a per-root total equals the sum of its per-surah counts
A cross-table consistency check on a different pair of exports.

In [ ]:
per_surah_sum = by_surah.groupby("root")["count"].sum()
merged = roots.set_index("root")["totalCount"].to_frame().join(per_surah_sum.rename("fromSurahs"))
bad = merged[merged["totalCount"] != merged["fromSurahs"]]
assert bad.empty, bad.head()
print("OK: all", len(merged), "roots agree between root-frequencies and root-surah-counts")

## Check 5: the extended association measures recompute from their own counts
The `association-extended` table (from v1.4.0) gives each pair's shared verses and both roots' verse counts. logDice and deltaP follow from those three numbers alone, so they can be recomputed here without the corpus. The table must also line up row for row with `association-pairs`.

In [ ]:
import numpy as np
names = tar.getnames()
if "association-extended.csv" not in names:
    print("SKIP: this pack predates the association-extended table")
else:
    ext = read("association-extended.csv")
    pairs = read("association-pairs.csv")
    assert len(ext) == len(pairs)
    assert (ext[["rootA", "rootB", "sharedVerses"]].values == pairs[["rootA", "rootB", "sharedVerses"]].values).all()
    N = 6236
    k11, fA, fB = ext["sharedVerses"], ext["versesA"], ext["versesB"]
    log_dice = 14 + np.log2(2 * k11 / (fA + fB))
    delta_p = k11 / fA - (fB - k11) / (N - fA)
    assert np.allclose(log_dice, ext["logDice"], atol=0.0005 + 1e-9), (log_dice - ext["logDice"]).abs().max()
    assert np.allclose(delta_p, ext["deltaP_BgivenA"], atol=0.00005 + 1e-9), (delta_p - ext["deltaP_BgivenA"]).abs().max()
    print("OK:", len(ext), "pairs; logDice and deltaP recompute within the table's rounding")

## Check 6: surah distances recompute from the per-surah root counts
The `surah-distance` table (from v1.5.0) gives each surah pair's Jensen-Shannon distance. It follows from the two surahs' root counts alone, which `root-surah-counts` carries, so it can be recomputed here without the corpus. The permutation null cannot be replayed without the site's seeded generator, so p and q are not checked here; the distance, the token counts and the excess arithmetic are.

In [ ]:
if "surah-distance.csv" not in tar.getnames():
    print("SKIP: this pack predates the surah-distance table")
else:
    dist = read("surah-distance.csv")
    counts = by_surah.pivot_table(index="root", columns="surah", values="count", aggfunc="sum", fill_value=0)

    def jsd_distance(a, b):
        p, q = a / a.sum(), b / b.sum()
        m = (p + q) / 2
        def kl(x):
            nz = x > 0
            return (x[nz] * np.log2(x[nz] / m[nz])).sum()
        return float(np.sqrt(max(0.0, 0.5 * kl(p) + 0.5 * kl(q))))

    assert len(dist) == 114 * 113 // 2, len(dist)
    sample = dist.iloc[::97]            # 67 pairs spread over the table
    for r in sample.itertuples(index=False):
        a, b = counts[r.surahA].values.astype(float), counts[r.surahB].values.astype(float)
        assert int(a.sum()) == r.rootedTokensA and int(b.sum()) == r.rootedTokensB, (r.surahA, r.surahB)
        assert abs(jsd_distance(a, b) - r.distance) < 6e-5, (r.surahA, r.surahB, jsd_distance(a, b), r.distance)
    # excess = distance - nullMean, to the table's four decimals
    assert ((dist["distance"] - dist["nullMean"] - dist["excess"]).abs() < 1.5e-4).all()
    print("OK:", len(sample), "of", len(dist), "surah distances recomputed from root counts; excess = distance - nullMean")

## Check 7: period keyness G2 recomputes from the root counts
The `keyness-period` table (from v1.6.0) gives each root's count in a revelation period and elsewhere. Dunning's G2 follows from those and the period token totals, so it can be recomputed here. The permutation p-values are the site's 100,000 seeded draws and are not replayed; the count of passing tests is checked against the published figure instead.

In [ ]:
if "keyness-period.csv" not in tar.getnames():
    print("SKIP: this pack predates the keyness-period table")
else:
    key = read("keyness-period.csv")
    total_tokens = int(surahs["tokenCount"].sum())
    period_tokens = surahs.groupby("period")["tokenCount"].sum().to_dict()
    root_total = roots.set_index("root")["totalCount"]

    def g2(a, b, c, d):
        n = a + b + c + d
        cells = [(a, (a + b) * (a + c) / n), (b, (a + b) * (b + d) / n), (c, (c + d) * (a + c) / n), (d, (c + d) * (b + d) / n)]
        return 2 * sum(o * np.log(o / e) for o, e in cells if o > 0)

    assert len(key) == 4 * len(roots)
    # the table's universe is all tokens, so a + b is the root's rooted total
    assert (key["count"] + key["countRest"] == key["root"].map(root_total)).all()
    sample = key.iloc[::53]
    for r in sample.itertuples(index=False):
        a, b = int(r.count), int(r.countRest)
        c = period_tokens[r.period] - a
        d = total_tokens - period_tokens[r.period] - b
        assert abs(g2(a, b, c, d) - r.g2) < 6e-4, (r.root, r.period)
    PUBLISHED_PASSING = 132       # data/keyness/period-keyness.json observed.passingBH
    assert int((key["bhQ"] <= 0.05).sum()) == PUBLISHED_PASSING, int((key["bhQ"] <= 0.05).sum())
    print("OK:", len(sample), "G2 values recomputed;", PUBLISHED_PASSING, "of", len(key), "root-period tests pass BH at q <= 0.05")

## Check 8: period joint counts add up to the published shared verses
The `association-by-period` table (from v1.6.0) splits each pair's shared verses over the four periods. They must sum to `association-pairs.sharedVerses` for every pair, and Cochran's Q must be non-negative.

In [ ]:
if "association-by-period.csv" not in tar.getnames():
    print("SKIP: this pack predates the association-by-period table")
else:
    abp = read("association-by-period.csv")
    periods = ["meccan-early", "meccan-middle", "meccan-late", "medinan"]
    joined = abp.merge(pairs[["rootA", "rootB", "sharedVerses"]], on=["rootA", "rootB"], how="left")
    assert len(abp) == len(pairs) and joined["sharedVerses"].notna().all()
    assert (joined[[f"shared_{p}" for p in periods]].sum(axis=1) == joined["sharedVerses"]).all()
    assert (abp["heterogeneityQ"].dropna() >= 0).all()
    print("OK: period joint counts sum to sharedVerses for all", len(abp), "pairs;",
          int((abp["heterogeneityBhQ"] <= 0.05).sum()), "differ across periods at BH q <= 0.05")

### What this does and does not show
It shows the published figures are internally consistent and recomputable from the pack.
It does not validate the Leeds annotation itself: every number inherits that corpus's root assignments.